# TD6 : Vision Language Models

In this TD, we will implement a Vision Language Model and finetune one for a Visual Question Answering task.

We will implement every block that makes a VLM from scratch: a vision encoder, a text decoder (both based on the transformer architecture), and a multimodal projector. We will try to create a deep understanding of what is happening.
Here is a diagram for the transformer architecture:

<img src="https://raw.githubusercontent.com/AviSoori1x/seemore/refs/heads/main/images/vlm.png" width=512> 

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
import math
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from einops import rearrange, repeat
from PIL import Image
from torchvision import transforms
import requests
from io import BytesIO
from torchvision.models.feature_extraction import get_graph_node_names, create_feature_extractor
import timm

## 1.1 The Vision Encoder 

We first implement the vision encoder, based on the Vision Transformer (ViT) architecture introduced in 2020 in the paper [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929). 

To apply the transformer to images, we need to transform an image into tokens. The approach introduced in ViT is to cut the image into patches that are then transformed into a token trhought a linear projection.

<img src="https://viso.ai/wp-content/uploads/2021/09/vision-transformer-vit.png" width=768> 

#### Question 1 
Implement the PatchEmbedding class that allows to transform an image into a sequence of patches. 

Hint: Use a Conv2D with the right kernel size and stride to do the linear projection of non-overlapping patches

In [ ]:
class PatchEmbeddings(nn.Module):
    def __init__(self, img_size=96, patch_size=16, hidden_dim=512):
        super().__init__()
        # To complete
        self.imag_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        
        #liner projection using Conv2d
        self.projection = nn.Conv2d(3, hidden_dim, kernel_size=patch_size, stride=patch_size)
        pass
            
    def forward(self, x):
        # To complete
        # x:[B,3,H,W]
        x = self.projection(x)  # [B, hidden_dim, grid_size, grid_size]
        x = x.flatten(2)  # [B, hidden_dim, num_patches]
        x = x.transpose(1, 2)  # [B, num_patches, hidden_dim]
        return x

Next, we implement the key components of the trasnformer architecture. We implement the  Multi-Head Attention, a LayerNorm and a Multilayer Perception. 

#### The Attention Mechanism 

The idea behind attention is the following. Imagine you want to retrieve information from a dictionary. The dictionnary is indexed by keys which maps to a particular value. Now, you have a query which will be matched against the keys of the dict and if you have a match, you will retrieve the associated value.
Attention is very similar to this simple retrieval example. Now, with real data, we don't have this structure, we however are going to learn to create it. 

We have 2 sets of vectors (also named tokens). One is $X_{to}$ which is the destination set. We want to be able to map this set of tokens to queries. We achieve this by doing a linear projection of $X_{to}$ to obatain:  $Q = W_QX_{to}$

The other set is $X_{from}$ the set from which we want to retrieve information. We will need to extract both keys and values from this set. We therefore do 2 linear projections of $X_{from}$ to obtain:  $K = W_KX_{from}$ and $V = W_VX_{from}$.

Now, contrary to the dictionnary where queries and values are exact matchs, we don't have this here. Therefore, we will perform a softer match by computing the similarity matrix between $Q$ and $K$. Then for each $Q$, we want to output the values that have the higher similarity. We therefore output the weighted sum of the values, weighted by the softmax of the similarity (also called the attention matrix).

Finally, the attention operation is given by the cross attention:

$$
A(Q,K,V) = \text{SoftMax}(\frac{QK^T}{\sqrt{d_k}})V
$$

We divide the similarity by $\sqrt{d_k}$ for stability reason to avoid the similarity to explode with big vectors which would lead to very sharp attention coeficients.

#### Question 2 
Implement the attention operation, use  `torch.einsum` to easily compute the similarity matrix.

In [3]:
class Attention(nn.Module):
    def __init__(self, x_to_dim, x_from_dim, hidden_dim):
        super(Attention, self).__init__()
        # To complete
        self.query = nn.Linear(x_to_dim, hidden_dim)
        self.key = nn.Linear(x_from_dim, hidden_dim)
        self.value = nn.Linear(x_from_dim, hidden_dim)
        self.scale = math.sqrt(hidden_dim)
        
        pass
        
    def forward(self, x_to, x_from):
        # x_to = [batch size, x_to_len, x_to_dim]
        # x_from = [batch size, x_from_len, x_from_dim]
        query = self.query(x_to)  # [batch size, x_to_len, hidden_dim]
        key = self.key(x_from)  # [batch size, x_from_len, hidden_dim]
        value = self.value(x_from)  # [batch size, x_from_len, hidden_dim]
        
        #similarity matrix using torch.einsum
        # b: batch, i: query tokens, j: key tokens, k: hidden dim
        sim = torch.einsum('bik,bjk->bij', query, key)/self.scale  # [batch size, x_to_len, x_from_len]
        attn = F.softmax(sim, dim=-1)  # [batch size, x_to_len, x_from_len]
        # weighted sum of value
        out = torch.einsum('bij,bjk->bik', attn, value)  # [batch size, x_to_len, hidden_dim]
        # To complete
        return out

### Multi-head attention

We improve the above attention implementation by introducing mult-head attention. The idea here is that we compute the attention on subspaces of the $Q,K,V$ triplets. 
We split each vector in $n$ subsets and compute the attention for each subset. At the end, we concatenate every attention output and project it with an output projection.

#### Question 3
Implement Multihead attention.

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, x_to_dim, x_from_dim, hidden_dim, n_heads):
        super(MultiHeadAttention, self).__init__()
        # To complete
        assert hidden_dim % n_heads == 0# "hidden_dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = hidden_dim // n_heads
        
        self.query = nn.Linear(x_to_dim, hidden_dim)
        self.key = nn.Linear(x_from_dim, hidden_dim)
        self.value = nn.Linear(x_from_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        pass

    def forward(self, x_to, x_from):
        # x_to = [batch size, x_to_len, x_to_dim]
        # x_from = [batch size, x_from_len, x_from_dim]
        b, n_to, _ = x_to.shape
        b, n_from, _ = x_from.shape
        
        q = self.q(x_to).view(b, n_to, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k(x_from).view(b, n_from, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v(x_from).view(b, n_from, self.n_heads, self.head_dim).transpose(1, 2) 

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(attn, dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(b, n_to, -1)

        # To complete
        return self.out_proj(out)

MultiheadAttention is the attention that is used in transformers in pratice. It is used in 2 flavors:
- Self Attention: When $X_{to}$ attends itself ($X_{to}=X_{from}$)
- Cross Attention. $X_{to}\neq X_{from}$

#### Question 4
Implement MultiHead Self Attention

In [7]:
class MultiHeadSelfAttention(MultiHeadAttention):
    def __init__(self, x_dim, hidden_dim, n_heads):
        super(MultiHeadSelfAttention, self).__init__(x_dim, x_dim, hidden_dim, n_heads)
        # To complete   
        pass

    def forward(self, x):
        # x = [batch size, x_len, x_dim]
        # To complete
        return super().forward(x, x)
        pass

### LayerNorm
Normalizing the output of a deep learning layer helps a lot with convergence and stability. 
Until Transformers, the most used normalization is BatchNorm. We normalize the data among the batch dimension. However, this has a few problems.
- The normalization depend on the other samples in the batch
- When using multiple GPUs, BatchNorm needs to synchronize the batch statistic across GPUs, which locks the forward process and slow down training.

The last element is the most important one. Transformers, aims to be a easy to parralilize architecture and can't afford to use batchnorm.

Instead, Transformers uses Layer Norm. LayerNorm is sample dependent, which removes the synchronization issue. We normalize over the channel dimension instead of the batch dimension.

<img src="https://production-media.paperswithcode.com/methods/Screen_Shot_2020-05-19_at_4.24.42_PM.png" width=512>

To account for the loss of capacity, we map the output by a linear transformation with a learned bias and scale.

#### Question 5
Implement the LayerNorm

In [8]:
class LayerNorm(nn.Module):
    # To complete
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta    
    pass

### Feed Feedward Network

Finally, the last block is a Multilayer Perceptron, also called feed-forward network (FFN), with one hidden layer. This layer has usually a size of $4 * input\_dim$. This is followed by a dropout layer and an activation function. Here, we will use [GELU](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html).

#### Question 6
Implement the FFN layer

In [9]:
class FFN(nn.Module):
    def __init__(self, n_embd, dropout=0.1, expansion_factor=4):
        super().__init__()
        # To complete
        self.net = nn.Sequential(
            nn.linear(n_embd, expansion_factor * n_embd),
            nn.GELU(),
            nn.Linear(expansion_factor * n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)
        pass

### Transformer block

To complete the Transformer Encoder block, the last thing that we are missing are the skip connection. Like in ResNet, the transformer architecture implements the skip-connection. This allow for a better gradient flow avoiding vanishing gradient.
There is a skip connection after the attention and the feed forward network. 

#### Question 7
Given at the transformer figure at the top (right-hand side), implement the Transformer Encoder Block

In [10]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, data_dim, hidden_dim, n_heads, dropout_rate=0.1):
        super().__init__()
        self.ln1 = LayerNorm(data_dim)
        self.ln2 = LayerNorm(hidden_dim)
        self.attn = MultiHeadSelfAttention(data_dim, hidden_dim, n_heads)
        self.ffn = FFN(hidden_dim, dropout_rate)
        # To complete
        pass
    
    def forward(self, x):
        # x = [batch size, x_len, hidden dim]
        # To complete
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
        pass

### Positional Embedding

We have implemented the Patch Embedding, the Multi-Head Self Attention, the Transformer Encoder Block. The last component remainining is the Positional Embedding (see left-hand side of the ViT figure). 

The transformers architecture is permutation independent. That means that for every token, we can swap 2 tokens and have the exact same result. However, the position of the token is a very important information to consider. If a pixel (or a patch) is nearby another pixel, we want the transformer to be able to capture such information. Which is not the case for now.
That's why we introduce positional encodings. For each token, add the positional encoding to the original token:

$$
X_i = X_i + PE(i)
$$

with X_i the token at the i dimension.

A common positional encodings is the learned positional encoding. Simply, we let the network learn a set of tensor $PE$ that match the sequence length and dimension of the tokens.

Note: Another widely used positionial embedding is the [Sinuoidal Positional Embedding](https://arxiv.org/abs/2407.09370)

#### Question 8
Implement the Learned Positional Embedding

In [11]:
class LearnedPositionalEncoding(nn.Module):
    def __init__(self, hidden_dim, max_len):
        # To complete
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, max_len, hidden_dim))
        pass
    
    def forward(self, x):
        # x = [batch size, seq len, hidden dim]
        # To complete
        return x + self.pe[:, :x.size(1), :]
        pass

## Putting the vision encoder together 

Now we have have everything we need to implement the Vision Transformer.

We also add an extra token, known as the classification token, that will be the token which will be use to predict upon. After going through the N transformer layers, this is the token that is outputted. 

#### Question 9
Given the ViT architecture on the top figure, implement the ViT


In [12]:
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, hidden_dim, n_heads, n_layers, dropout_rate):
        super(ViT, self).__init__()
        # To complete
        self.patch_embed = PatchEmbeddings(img_size, patch_size, hidden_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        
        num_patches = (img_size // patch_size) ** 2
        self.pos_embed = LearnedPositionalEncoding(hidden_dim, num_patches + 1)
        
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(hidden_dim, hidden_dim, n_heads, dropout_rate)
            for _ in range(n_layers)
        ])
        
        pass

    def forward(self, X):
        # x = [batch size, 3, image height, image width]
        # To complete
        x = self.patch_embed(X)  # [batch size, num_patches, hidden_dim]
        b, n, _ = x.shape
        
        self.cls_token = self.cls_token.expand(b, -1, -1)  # [batch size, 1, hidden_dim]
        x = torch.cat((self.cls_token, x), dim=1)  # [batch size, num_patches + 1, hidden_dim]
        x = self.pos_embed(x)  # [batch size, num_patches + 1, hidden_dim]
        
        for layer in self.layers:
            x = layer(x)
            
        return self.ln(x[:, 0])  # [batch size, hidden_dim]
        pass

## 1.2 The Multimodal Projector 

Overall, the ViT class takes an input image and returns the embedding corresponding to the class token (CLS), which is then used to condition the text generation in the language decoder. 

However, we can not directly concatenate this to the text embeddings. We need to project this from the dimensionality of image embeddings from the vision transformer to the dimensionality of text embeddings. This is done by the multimodal projector.

This projector is usually a single learnable layer followed by a non-linearity or an MLP. Here we will implement a MLP wit one hidden layer, an expansion factor of $4$ and a GELU activation function. 

#### Question 10
Implement the Multimodal Projector

In [15]:
class MultiModalProjector(nn.Module):
    # To complete
    def __init__(self,vision_dim, text_dim, expansion_factor = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vision_dim + text_dim, expansion_factor *vision_dim),
            nn.GELU(),
            nn.Linear(expansion_factor * vision_dim, text_dim)
        )
    def forward(self,x):
        return self.net(x)
    pass


## 1.3 The Language Transformer Decoder

The final component we need to look at is the decoder language model. 
The (text) Transformer Decoder is similar to the (Vision) Transformer Encoder defined before. The main differences are: 
- A text token embedding replaces the patch embedding. 
- Causal Self Attention replaces Self Attention in the Transformer Block. 
- A language modeling head is added on top of the last Transformer Block. 


### Causal Attention

In Causal Attention, masking is applied in each attention head to obscure any information following the current token's position, thereby directing the model's attention to only the preceding parts of the sequence. A token can not attend to the "future" (following) tokens in the sentence. 

In practice, a lower triangular mask $M$ is added to the similarity matrix between $Q$ and $K$. 

The causal attention operation is then given by: 
$$
A_{causal}(Q,K,V) = \text{SoftMax}(\frac{QK^T + M}{\sqrt{d_k}})V
$$

<img src="https://blog.sailor.plus/deep-learning/images/1613723693323.png" width=512> 


#### Question 11 
Implement the Causal Multi Head Self Attention 

Hint: You can start from the class implemented before. 

In [16]:
class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(self, x_dim, hidden_dim, n_heads):
        # To complete
        super().__init__()
        self.n_heads = n_heads
        self.mha = MultiHeadSelfAttention(x_dim, hidden_dim, n_heads)
        pass

    def forward(self, x):
        # x = [batch size, x_len, x_dim]
        # To complete
        b, n, d = x.shape
        #create causal mask
        mask = torch.tril(torch.ones(n, n)).to(x.device)  # [x_len, x_len]
        query = self.mha.query(x).view(b, n, self.n_heads, -1).transpose(1, 2)
        key = self.mha.key(x).view(b, n, self.n_heads, -1).transpose(1, 2)
        value = self.mha.value(x).view(b, n, self.n_heads, -1).transpose(1, 2)
        
        sim = torch.einsum('bhid,bhjd->bhij', query, key) / math.sqrt(self.mha.head_dim)
        sim = sim.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(sim, dim=-1)
        
        out = (attn @ value).transpose(1, 2).reshape(b, n, -1)
        return self.mha.out_proj(out)

### Transformer Decoder Block

Now we implement a Transformer Decoder Block using the Causal Attention. 

#### Question 12

Implement the Transformer Decoder Block 



In [17]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, data_dim, hidden_dim, n_heads, dropout_rate=0.1):
        # To complete
        super().__init__()
        self.ln1 = LayerNorm(data_dim)
        self.attn = CausalMultiHeadSelfAttention(data_dim, hidden_dim, n_heads)
        self.ln2 = LayerNorm(hidden_dim)
        self.ffn = FFN(hidden_dim, dropout_rate)
        pass
    
    def forward(self, x):
        # To complete
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
        pass

## Building the Language Transformer Decoder

#### Question 13

Implement the Language Transformer Decoder

In [18]:
class LanguageTransformerDecoder(nn.Module):
    def __init__(self, n_embd, image_embed_dim, vocab_size, n_heads, n_layers):
        # To complete
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, 1024, n_embd))
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(n_embd, n_embd, n_heads) for _ in range(n_layers)
        ])
        self.head = nn.Linear(n_embd, vocab_size)
        pass

    def forward(self, idx, image_embeds):
        # To complete
        # idx: [batch, seq_len], image_embeds: [batch, n_embd]
        x = self.tok_emb(idx)
        # Prepend the image embedding to the sequence
        x = torch.cat((image_embeds.unsqueeze(1), x), dim=1)
        x = x + self.pos_emb[:, :x.size(1), :]
        
        for layer in self.layers:
            x = layer(x)
            
        return self.head(x)
        pass
    
    def generate(self, idx, image_embeds, max_new_tokens):
        ### Do not complete this method ### 
        # With the logits outputted by the forward method 
        # and using the sampling methods seen in TD5, 
        # we can generate some text tokens!! 
        pass

With the logits outputted by the forward method and using the sampling methods seen in TD5, we can generate some text tokens!

## 1.4 Bringing everything together to implement the Vision Language Model

Now, we have all the element to build our Vision Languauge model. 

#### Question 14 

Implement the Vision Language Model class

In [19]:
class VisionLanguageModel(nn.Module):
    def __init__(
            self,
            n_embd,
            image_embed_dim,
            vocab_size,
            n_enc_layers,
            img_size, patch_size,
            n_heads,
            n_dec_layers,
            dropout_rate
        ):
        # To complete
        super().__init__()
        self.encoder = ViT(img_size, patch_size, image_embed_dim, n_heads, n_enc_layers, dropout_rate)
        self.projector = MultiModalProjector(image_embed_dim, n_embd)
        self.decoder = LanguageTransformerDecoder(n_embd, image_embed_dim, vocab_size, n_heads, n_dec_layers)
        pass

    def forward(self, img_array, idx):
        # To complete
        vis_feat = self.encoder(img_array)
        proj_feat = self.projector(vis_feat)
        logits = self.decoder(idx, proj_feat)
        return logits
        pass

    def generate(self, img_array, idx, max_new_tokens):
        # To complete
        pass

## 2. Finetuning a VLM for Visual Question Answering task

Training a Vision Language Model from scratch requires a lot of training data and GPU ressources. Then, we are going to finetune an already pretrained VLM on a specific task: Visual Question Answering.  

We will fine-tune the recent VLM `Florence-2` from Microsoft on the dataset DocVQA.

Lets start by donwloading the model and the dataset. 

In [1]:
import transformers
transformers.__version__   

'4.49.0'

In [1]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoProcessor
import torch

data = load_dataset("HuggingFaceM4/DocumentVQA", split=["train[:10%]", "validation[:10%]", "test[:10%]"], cache_dir="/Data/Dacnguyen")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-base-ft", trust_remote_code=True, revision='refs/pr/6').to(device)
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-base-ft", trust_remote_code=True, revision='refs/pr/6')

torch.cuda.empty_cache()

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

/users/eleves-a/2024/dac.nguyen/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Florence2LanguageForConditionalGeneration has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `Pr

Let's do inference with our dataset first to see how the model performs before fine-tuning.

In [ ]:
# Function to run the model on an example
def run_example(task_prompt, text_input, image):
    prompt = task_prompt + text_input

    # Ensure the image is in RGB mode
    if image.mode != "RGB":
        image = image.convert("RGB")

    inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        num_beams=3
    )
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed_answer = processor.post_process_generation(generated_text, task=task_prompt, image_size=(image.width, image.height))
    return parsed_answer

In [ ]:
for idx in range(3):
    print(run_example("DocVQA", 'What is written on top of the document?', data['train'][idx]['image']))
    display(data['train'][idx]['image'].resize([350, 350]))

We need to construct our dataset. Note how we are adding a new task prefix `<DocVQA>` before the question when constructing the prompt.

In [ ]:
from torch.utils.data import Dataset

class DocVQADataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        question = "<DocVQA>" + example['question']
        first_answer = example['answers'][0]
        image = example['image']
        if image.mode != "RGB":
            image = image.convert("RGB")
        return question, first_answer, image


Let's get to fine-tuning. We will instntiate our dataset, the data collator, and start training. 

In [ ]:
import os
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
from transformers import AutoProcessor, get_scheduler
from bitsandbytes.optim import AdamW

def collate_fn(batch):
    questions, answers, images = zip(*batch)
    inputs = processor(text=list(questions), images=list(images), return_tensors="pt", padding=True).to(device)
    return inputs, answers

# Create datasets
train_dataset = DocVQADataset(data['train'])
val_dataset = DocVQADataset(data['validation'])

Finetuning the model on the entire dataset will be too long for us (around 2,5 hours per epoch) and could be too heavy for the vRAM of the GPU we are using. We reduce the size of the dataset using `Subset`.

You can adapt the size of the subsets depending on your GPU. 

If you have time, run the finetuning on the entire dataset, the results will be even better! 

In [ ]:
# Use a subset of the dataset for training
train_dataset = Subset(train_dataset, list(range(0, 1000)))
val_dataset = Subset(val_dataset, list(range(0, 200)))

# Create DataLoader
batch_size = 2
num_workers = 0

train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers)

#### Question 15 

Complete the training loop in the `train model` function

In [ ]:
def train_model(train_loader, val_loader, model, processor, epochs=10, lr=1e-6):
    optimizer = AdamW(model.parameters(), lr=lr)
    num_training_steps = epochs * len(train_loader)
    lr_scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        i = -1
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch + 1}/{epochs}"):
            i += 1
            inputs, answers = batch

            input_ids = inputs["input_ids"]
            pixel_values = inputs["pixel_values"]
            labels = processor.tokenizer(text=answers, return_tensors="pt", padding=True, return_token_type_ids=False).input_ids.to(device)

            ### Complete here:  ###
            # Get the logits from the model
            outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            # Backpropagate
            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            ### End completion ###

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        print(f"Average Training Loss: {avg_train_loss}")

        # Validation phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch + 1}/{epochs}"):
                inputs, answers = batch

                input_ids = inputs["input_ids"]
                pixel_values = inputs["pixel_values"]
                labels = processor.tokenizer(text=answers, return_tensors="pt", padding=True, return_token_type_ids=False).input_ids.to(device)

                outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
                loss = outputs.loss

                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"Average Validation Loss: {avg_val_loss}")

        # Save model checkpoint
        output_dir = f"./model_checkpoints/epoch_{epoch+1}"
        os.makedirs(output_dir, exist_ok=True)
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)


We will freeze image encoder for this TP. The authors have reported improvement in unfreezing image encoder, but note that this will result in more resource usage.

In [ ]:
for param in model.vision_tower.parameters():
    param.is_trainable = False

Note: if the following cell crash with the error `OutOfMemoryError: CUDA out of memory.`, try to reduce the batch size and/or the number of example in the train/validation set. 

In [ ]:
train_model(train_loader, val_loader, model, processor, epochs=2)

Let's do inference with our finetuned model: 

In [ ]:
for idx in range(3):
    print(run_example("DocVQA", 'What is written on top of the document?', data['train'][idx]['image']))
    display(data['train'][idx]['image'].resize([350, 350]))